In [4]:
"""
Python Basics Quiz Game
-----------------------
A multiple-choice quiz with scoring, feedback, and a saved leaderboard.

Concepts practiced:
  * Data structures: a list of dictionaries (each holding a nested dictionary)
  * Loops: while (menu, input validation) and for (questions, leaderboard)
  * String manipulation: strip, upper, title, split, formatting
  * File I/O: reading and writing the leaderboard file

To customize: change TOPIC and edit the QUESTIONS list below.
"""

import random

LEADERBOARD_FILE = "leaderboard.txt"
MAX_LEADERBOARD_ENTRIES = 5
TOPIC = "Python Basics"

# Each question is a dictionary. "choices" is a nested dictionary mapping
# a letter to the answer text.
QUESTIONS = [
    {
        "question": "Which function prints text to the screen?",
        "choices": {"A": "show()", "B": "print()", "C": "display()", "D": "echo()"},
        "answer": "B",
        "explanation": "print() sends output to the console.",
    },
    {
        "question": "What is the result of 7 // 2 ?",
        "choices": {"A": "3.5", "B": "4", "C": "3", "D": "2"},
        "answer": "C",
        "explanation": "// is floor division: it drops the decimal part.",
    },
    {
        "question": "Which of these is a valid list?",
        "choices": {"A": "[1, 2, 3]", "B": "(1, 2, 3)", "C": "{1, 2, 3}", "D": "<1, 2, 3>"},
        "answer": "A",
        "explanation": "Square brackets create a list. () is a tuple, {} is a set.",
    },
    {
        "question": "What does len('hello') return?",
        "choices": {"A": "4", "B": "5", "C": "6", "D": "Error"},
        "answer": "B",
        "explanation": "len() counts characters, and 'hello' has 5.",
    },
    {
        "question": "Which keyword starts a loop that runs while a condition is True?",
        "choices": {"A": "for", "B": "loop", "C": "repeat", "D": "while"},
        "answer": "D",
        "explanation": "while repeats its block as long as the condition stays True.",
    },
    {
        "question": "How do you get a dictionary's value for the key 'name'?",
        "choices": {"A": "d['name']", "B": "d.name", "C": "d(name)", "D": "d->name"},
        "answer": "A",
        "explanation": "Dictionaries are indexed with square brackets and a key.",
    },
    {
        "question": "What is 'python'.upper() ?",
        "choices": {"A": "Python", "B": "python", "C": "PYTHON", "D": "pYTHON"},
        "answer": "C",
        "explanation": "upper() returns a new string with every letter capitalized.",
    },
    {
        "question": "Which symbol starts a comment in Python?",
        "choices": {"A": "//", "B": "#", "C": "--", "D": "/*"},
        "answer": "B",
        "explanation": "Everything after # on a line is ignored by Python.",
    },
]


# ---------- Leaderboard (file reading / writing) ----------

def load_leaderboard():
    """Read the leaderboard file and return a list of (name, score) tuples."""
    entries = []
    try:
        with open(LEADERBOARD_FILE, "r") as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                # rsplit from the right so a name containing a comma still works
                name, score_text = line.rsplit(",", 1)
                if score_text.isdigit():
                    entries.append((name, int(score_text)))
    except FileNotFoundError:
        pass  # first run: no file yet, so the leaderboard is just empty
    return entries


def save_score(name, score):
    """Add a new score, keep only the top entries, and write them back."""
    entries = load_leaderboard()
    entries.append((name, score))
    entries.sort(key=lambda entry: entry[1], reverse=True)
    entries = entries[:MAX_LEADERBOARD_ENTRIES]

    with open(LEADERBOARD_FILE, "w") as file:
        for entry_name, entry_score in entries:
            file.write(f"{entry_name},{entry_score}\n")


def show_leaderboard():
    """Print the saved high scores."""
    entries = load_leaderboard()
    print("\n" + "=" * 32)
    print("        LEADERBOARD")
    print("=" * 32)
    if not entries:
        print("  No scores yet. Be the first!")
    else:
        for rank, (name, score) in enumerate(entries, start=1):
            print(f"  {rank}. {name:<15} {score:>3} pts")
    print("=" * 32)


# ---------- Quiz logic ----------

def get_player_name():
    """Keep asking until the player enters a non-empty name."""
    while True:
        name = input("Enter your name: ").strip().replace(",", "")
        if name:
            return name.title()
        print("Name can't be empty. Try again.")


def ask_question(number, total, item):
    """Show one question, validate the answer, and return True if correct."""
    print(f"\nQuestion {number} of {total}")
    print(item["question"])
    for letter, text in item["choices"].items():
        print(f"  {letter}) {text}")

    while True:
        guess = input("Your answer: ").strip().upper()
        if guess in item["choices"]:
            break
        valid = ", ".join(item["choices"])
        print(f"Please enter one of: {valid}")

    if guess == item["answer"]:
        print("Correct! +1 point")
        return True

    correct_text = item["choices"][item["answer"]]
    print(f"Not quite. The answer was {item['answer']}) {correct_text}")
    print(f"Hint: {item['explanation']}")
    return False


def get_rating(score, total):
    """Return a message based on the percentage score."""
    percent = score / total * 100
    if percent == 100:
        return "Perfect score! You're a pro."
    if percent >= 70:
        return "Great job!"
    if percent >= 40:
        return "Nice try. Keep practicing!"
    return "Keep studying, you'll get there!"


def play_game():
    """Run one full round of the quiz."""
    name = get_player_name()
    questions = QUESTIONS[:]      # copy so we don't change the original list
    random.shuffle(questions)
    total = len(questions)
    score = 0

    print(f"\nGood luck, {name}! {total} questions on {TOPIC}.")

    for number, item in enumerate(questions, start=1):
        if ask_question(number, total, item):
            score += 1

    print("\n" + "-" * 32)
    print(f"Final score: {score}/{total}")
    print(get_rating(score, total))
    save_score(name, score)
    show_leaderboard()


def main():
    """Main menu loop."""
    print(f"Welcome to the {TOPIC} Quiz!")
    while True:
        print("\n1) Play  2) Leaderboard  3) Quit")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            play_game()
        elif choice == "2":
            show_leaderboard()
        elif choice == "3":
            print("Thanks for playing. Goodbye!")
            break
        else:
            print("Please enter 1, 2, or 3.")


if __name__ == "__main__":
    main()
    

Welcome to the Python Basics Quiz!

1) Play  2) Leaderboard  3) Quit


Choose an option:  1
Enter your name:  amy



Good luck, Amy! 8 questions on Python Basics.

Question 1 of 8
Which symbol starts a comment in Python?
  A) //
  B) #
  C) --
  D) /*


Your answer:  b


Correct! +1 point

Question 2 of 8
What is the result of 7 // 2 ?
  A) 3.5
  B) 4
  C) 3
  D) 2


...

Correct! +1 point

Question 8 of 8
Which keyword starts a loop that runs while a condition is True?
  A) for
  B) loop
  C) repeat
  D) while


Your answer:  d


Correct! +1 point

--------------------------------
Final score: 8/8
Perfect score! You're a pro.

        LEADERBOARD
  1. Amy               8 pts
  2. Akshita           6 pts

1) Play  2) Leaderboard  3) Quit


Choose an option:  3


Thanks for playing. Goodbye!
